
# Stack Overflow Tag Prediction with Sampling


The notebook assumes a project layout like:

```text
project-root/
├── data/
│   ├── raw/
│   └── processed/
├── models/
├── notebooks/
└── backend/
```



## 1. Setup

Use the notebook from the project root so the relative paths work correctly.

If you do not already have the package for MLkNN, install it once in your environment:
`scikit-multilearn`


In [ ]:

from pathlib import Path
from collections import Counter
import random
import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import ToktokTokenizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    jaccard_score,
    precision_score,
    recall_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import NearestNeighbors

# Optional multilabel baseline
from skmultilearn.adapt import MLkNN

warnings.filterwarnings("ignore")

# Notebook appearance
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True

# NLTK data
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

tokenizer = ToktokTokenizer()
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words("english"))
PUNCT = '!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~'

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

QUESTIONS_FILE = RAW_DIR / "Questions.csv.zip"
TAGS_FILE = RAW_DIR / "Tags.csv.zip"

# Adjust this if you want to keep a larger or smaller tag vocabulary.
TOP_N_TAGS = 100



## 2. Load and merge the dataset

The original Stack Overflow Kaggle sample stores questions and tags separately.
This section rebuilds the combined table used by the model.


In [ ]:
def get_feature_names(vectorizer):
    if hasattr(vectorizer, "get_feature_names_out"):
        return vectorizer.get_feature_names_out()
    return vectorizer.get_feature_names()

questions = pd.read_csv(QUESTIONS_FILE, encoding="ISO-8859-1")
tags = pd.read_csv(TAGS_FILE, encoding="ISO-8859-1", dtype={"Tag": str})

tags["Tag"] = tags["Tag"].astype(str)
grouped_tags = tags.groupby("Id")["Tag"].apply(lambda x: " ".join(x))
grouped_tags = grouped_tags.reset_index(name="Tags")

df = questions.drop(columns=[c for c in ["OwnerUserId", "CreationDate", "ClosedDate"] if c in questions.columns])
df = df.merge(grouped_tags, on="Id", how="inner")

# Keep higher-quality questions and remove duplicates.
df = df[df["Score"] > 1].drop_duplicates(["Title", "Body", "Tags"]).copy()
df = df.drop(columns=[c for c in ["Id", "Score"] if c in df.columns])

df.head()



## 3. Tag exploration

- tag frequency analysis,
- selection of the most common tags,
- a filtered training set with the strongest label signal.


In [ ]:

df["Tags"] = df["Tags"].apply(lambda x: x.split())

all_tags = [tag for row in df["Tags"].values for tag in row]
tag_counts = Counter(all_tags)
most_common_tags = [tag for tag, _ in tag_counts.most_common(TOP_N_TAGS)]

print("Total unique tags:", len(set(all_tags)))
print(f"Top {TOP_N_TAGS} tags:", len(most_common_tags))

tag_freq_df = pd.DataFrame(tag_counts.most_common(TOP_N_TAGS), columns=["Tag", "Count"])
tag_freq_df.head(10)


In [ ]:

fig, ax = plt.subplots()
tag_freq_df.sort_values("Count", ascending=True).plot(
    kind="barh", x="Tag", y="Count", legend=False, ax=ax
)
ax.set_title(f"Top {TOP_N_TAGS} Tags")
ax.set_xlabel("Count")
ax.set_ylabel("Tag")
plt.tight_layout()
plt.show()


In [ ]:

def keep_common_tags(tags_list, common_tags):
    filtered = [tag for tag in tags_list if tag in common_tags]
    return filtered if filtered else None

df["Tags"] = df["Tags"].apply(lambda x: keep_common_tags(x, most_common_tags))
df = df.dropna(subset=["Tags"]).copy()
df["Tags"] = df["Tags"].apply(" ".join)
df["tag_count"] = df["Tags"].apply(lambda row: len(str(row).split(" ")))

print(df.shape)
df.head()



## 4. Text preprocessing



In [ ]:

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"'s", " ", text)
    text = re.sub(r"'ve", " have ", text)
    text = re.sub(r"can't", "can not ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"'re", " are ", text)
    text = re.sub(r"'d", " would ", text)
    text = re.sub(r"'ll", " will ", text)
    text = re.sub(r"\xa0", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def strip_punctuation(text):
    words = tokenizer.tokenize(text)
    regex = re.compile("[%s]" % re.escape(PUNCT))
    cleaned = []
    for word in words:
        if word in most_common_tags:
            cleaned.append(word)
        else:
            word = regex.sub(" ", word)
            if word not in most_common_tags:
                cleaned.append(word)
    cleaned = [item.strip() for item in cleaned if hasattr(item, "strip") and item.strip()]
    return " ".join(cleaned)

def remove_stopwords(text):
    words = tokenizer.tokenize(text)
    return " ".join([w for w in words if w not in STOP_WORDS])

def lemmatize_words(text):
    words = tokenizer.tokenize(text)
    return " ".join([lemmatizer.lemmatize(w, pos="v") for w in words])

def clean_digits(text):
    text = re.sub(r"\S*[0-9]+\S*", " ", text)
    text = re.sub(r"\b([a-zA-Z])\1+\b", " ", text)
    text = re.sub(r"\b\S*([a-zA-Z])\1{2,}\S*\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def final_clean(text):
    text = BeautifulSoup(str(text), "html.parser").get_text(" ")
    text = clean_text(text)
    text = clean_digits(text)
    text = strip_punctuation(text)
    text = remove_stopwords(text)
    text = lemmatize_words(text)
    text = re.sub(r"[^A-Za-z]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_question(title, body):
    question = f"{title} {body}"
    return final_clean(question)


In [ ]:

processed_rows = []
start = datetime.now()

for _, row in df.iterrows():
    processed_rows.append(
        {
            "question": build_question(row["Title"], row["Body"]),
            "tags": row["Tags"],
        }
    )

prepared_df = pd.DataFrame(processed_rows)
prepared_df["tag_count"] = prepared_df["tags"].apply(lambda x: len(str(x).split()))
print("Processed rows:", prepared_df.shape[0])
print("Time taken:", datetime.now() - start)

prepared_df.head()



## 5. Multilabel target preparation and vectorization

This section creates the multilabel target matrix and the TF-IDF representation used by all models below.


In [ ]:

X = prepared_df["question"]
y = prepared_df["tags"].apply(lambda x: x.split())

mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y)

count_vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_count = count_vectorizer.fit_transform(X)

tag_vectorizer = TfidfVectorizer(
    analyzer="word",
    min_df=2,
    max_df=1.0,
    strip_accents=None,
    encoding="utf-8",
    preprocessor=None,
    token_pattern=r"(?u)\S+",
    ngram_range=(1, 2),
    sublinear_tf=True,
)
X_tfidf = tag_vectorizer.fit_transform(X)

print("X_tfidf shape:", X_tfidf.shape)
print("y_bin shape:", y_bin.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_bin, test_size=0.2, random_state=0
)



## 6. Evaluation helpers


In [ ]:

def print_score(y_true, y_pred, clf_name="model"):
    print("CLF:", clf_name)
    print("Jaccard score: {:.4f}".format(jaccard_score(y_true, y_pred, average="samples") * 100))
    print("Subset accuracy: {:.4f}".format(accuracy_score(y_true, y_pred) * 100))
    print("Macro F1: {:.4f}".format(f1_score(y_true, y_pred, average="macro", zero_division=0) * 100))
    print("Micro F1: {:.4f}".format(f1_score(y_true, y_pred, average="micro", zero_division=0) * 100))
    print("Hamming loss: {:.4f}".format(hamming_loss(y_true, y_pred) * 100))
    print("Micro recall: {:.4f}".format(recall_score(y_true, y_pred, average="micro", zero_division=0) * 100))
    print(classification_report(y_true, y_pred, zero_division=0))
    print("-" * 100)

def average_jaccard(y_true, y_pred):
    return jaccard_score(y_true, y_pred, average="samples") * 100

def top_k_indices_from_proba(probabilities, k=5):
    return np.argsort(probabilities, axis=1)[:, -k:].tolist()

def matched(list1, list2):
    return len([value for value in list1 if value in set(list2)])

def evaluate_top_k(y_true_labels, y_pred_labels):
    precision, recall, f_measure = [], [], []
    for i in range(len(y_true_labels)):
        tp = matched(y_true_labels[i], y_pred_labels[i])
        pred_len = max(len(y_pred_labels[i]), 1)
        true_len = max(len(y_true_labels[i]), 1)
        precision.append(tp / pred_len)
        recall.append(tp / true_len)
        if precision[-1] + recall[-1] == 0:
            f_measure.append(0)
        else:
            f_measure.append((2 * precision[-1] * recall[-1]) / (precision[-1] + recall[-1]))
    return np.mean(precision), np.mean(recall), np.mean(f_measure)

y_true_labels = [np.where(row == 1)[0].tolist() for row in y_test]



## 7. Baseline model: One-vs-Rest Linear SVM

This is a strong baseline and a useful reference point before the sampling-based experiments.


In [ ]:

baseline_clf = OneVsRestClassifier(LinearSVC())
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)

print_score(y_test, baseline_pred, clf_name="OneVsRestClassifier(LinearSVC)")



## 8. Optional strong classifier: calibrated Linear SVM



In [ ]:

calibrated_clf = OneVsRestClassifier(CalibratedClassifierCV(LinearSVC()))
calibrated_clf.fit(X_train, y_train)
calibrated_pred = calibrated_clf.predict(X_test)

print_score(y_test, calibrated_pred, clf_name="OneVsRestClassifier(CalibratedClassifierCV(LinearSVC()))")

# Top-k ranking
probas = calibrated_clf.predict_proba(X_test)
y_pred_k = top_k_indices_from_proba(probas, k=5)
P, R, F1 = evaluate_top_k(y_true_labels, y_pred_k)

print(f"Top-5 precision: {P:.4f}")
print(f"Top-5 recall: {R:.4f}")
print(f"Top-5 F1: {F1:.4f}")



## 9. MLkNN

This section keeps the original MLkNN experiment, but isolates it as a standalone baseline.
Tune `k` for your dataset size and memory budget.


In [ ]:

mlknn_clf = MLkNN(k=200)
mlknn_clf.fit(X_train, y_train)
mlknn_pred = mlknn_clf.predict(X_test)

print_score(y_test, mlknn_pred, clf_name="MLkNN")



## 10. MLSMOTE



In [ ]:

def get_tail_label(df, ql=(0.05, 0.8)):
    irlbl = df.sum(axis=0)
    irlbl = irlbl[(irlbl > irlbl.quantile(ql[0])) & (irlbl < irlbl.quantile(ql[1]))]
    irlbl = irlbl.max() / irlbl
    threshold_irlbl = irlbl.median()
    return irlbl[irlbl > threshold_irlbl].index.tolist()

def get_minority_samples(X, y, ql=(0.05, 1.0)):
    tail_labels = get_tail_label(y, ql=ql)
    index = y[y[tail_labels].apply(lambda x: (x == 1).any(), axis=1)].index.tolist()
    X_sub = X[X.index.isin(index)].reset_index(drop=True)
    y_sub = y[y.index.isin(index)].reset_index(drop=True)
    return X_sub, y_sub

def nearest_neighbour(X, neigh):
    nbs = NearestNeighbors(n_neighbors=neigh, metric="euclidean", algorithm="kd_tree").fit(X)
    _, indices = nbs.kneighbors(X)
    return indices

def MLSMOTE(X, y, n_sample, neigh):
    indices2 = nearest_neighbour(X, neigh)
    n = len(indices2)

    new_X = np.zeros((n_sample, X.shape[1]))
    target = np.zeros((n_sample, y.shape[1]))

    for i in range(n_sample):
        reference = random.randint(0, n - 1)
        neighbor = random.choice(indices2[reference, 1:])
        all_point = indices2[reference]
        nn_df = y[y.index.isin(all_point)]
        ser = nn_df.sum(axis=0, skipna=True)
        target[i] = np.array([1 if val > 2.0 else 0 for val in ser])

        ratio = random.random()
        gap = abs(X.loc[reference, :] - X.loc[neighbor, :])
        new_X[i] = np.array(X.loc[reference, :] + ratio * gap)

    new_X = pd.DataFrame(new_X, columns=X.columns)
    target = pd.DataFrame(target, columns=y.columns)
    return new_X, target

X_train_df = pd.DataFrame.sparse.from_spmatrix(X_train, columns=get_feature_names(tag_vectorizer))
y_train_df = pd.DataFrame(y_train, columns=mlb.classes_)

tails = get_tail_label(y_train_df)
X_sub, y_sub = get_minority_samples(X_train_df, y_train_df)

X_res, y_res = MLSMOTE(X_sub, y_sub, n_sample=10000, neigh=5)

y_res["class_count"] = y_res.sum(axis=1)
no_class = y_res[y_res["class_count"] == 0.0].index.values.tolist()

if no_class:
    X_res.drop(no_class, axis=0, inplace=True)
    y_res.drop(no_class, axis=0, inplace=True)

y_res.drop(columns=["class_count"], inplace=True)

X_train_with_sampling = pd.concat([X_train_df, X_res], ignore_index=True)
y_train_with_sampling = pd.concat([y_train_df, y_res], ignore_index=True)

print("Before sampling:", y_train_df.shape)
print("After sampling:", y_train_with_sampling.shape)
print("Tail labels before:", len(tails))
print("Tail labels after:", len(get_tail_label(y_train_with_sampling)))



## 11. Classifier after MLSMOTE

re-training on the oversampled representation.


In [ ]:

X_train_os = X_train_with_sampling
y_train_os = y_train_with_sampling

# Convert sparse-like dataframe back to matrix form for the classifier.
X_train_os_matrix = X_train_os.astype(float).to_numpy()
y_train_os_matrix = np.asarray(y_train_os, dtype=int)

clf_after_mlsmote = OneVsRestClassifier(CalibratedClassifierCV(LinearSVC()))
clf_after_mlsmote.fit(X_train_os_matrix, y_train_os_matrix)

mlsmote_pred = clf_after_mlsmote.predict(X_test)
print_score(y_test, mlsmote_pred, clf_name="After MLSMOTE")



## 12. MLSOL




In [ ]:

class MLSol:
    def __init__(self):
        self.minority_class = {
            "SF": 0,
            "BD": 1,
            "RR": 2,
            "OT": 3,
            "MJ": 4,
        }
        self.k = None
        self.num_samples = None
        self.num_features = None
        self.num_classes = None
        self.knn_ = None
        self.C = None
        self.w = None
        self.T = None

    def oversample(self, X: np.ndarray, Y: np.ndarray, P: float, k: int):
        GenNum = int(np.ceil(len(X) * P))
        X_prime, Y_prime = X.copy(), Y.copy()
        self.k = k
        self.num_samples = len(X)
        self.num_features = X.shape[-1]
        self.num_classes = Y.shape[-1]
        self.knn_ = self._knn_majority(X, Y)
        self.C = self.get_C(Y)
        self.w = self.get_w(Y)
        self.T = self.InitTypes(Y)

        for _ in range(GenNum):
            seed_int, ref_int = self.seed_ref_instance()
            X_c, Y_c = self.GenerateInstance(seed_int, ref_int, X, Y)
            X_prime = np.vstack((X_prime, X_c))
            Y_prime = np.vstack((Y_prime, Y_c))

        return X_prime, Y_prime

    def get_C(self, Y: np.ndarray):
        C = np.zeros_like(Y)
        for idx, ind in self.knn_.items():
            Y_arr = Y[idx, :]
            knn_Y = Y[ind["knn_idx"], :]
            ind_ones = np.argwhere(Y_arr)[:, 0]
            temp_mat = np.zeros((self.k, self.num_classes))
            temp_mat[:, ind_ones] = 1
            C[idx, :] = (knn_Y != temp_mat).sum(axis=0)
        return C / self.k

    def get_w(self, Y: np.ndarray):
        C_less_1 = self.C < 1
        numerators = self.C * Y * C_less_1
        denominators = numerators.sum(axis=0)
        divide = np.ones_like(numerators)
        divide[:, :] = denominators
        w = (numerators / (divide + 1e-6)).sum(axis=1)
        return w / (w.sum() + 1e-6)

    def _knn_majority(self, X: np.ndarray, Y: np.ndarray):
        result = dict()
        for idx, row in enumerate(X):
            row = np.array([row] * self.num_samples)
            row -= X
            row = np.linalg.norm(row, ord=2, axis=1)
            ind_ = np.argsort(row)
            ind_ = ind_[ind_ != idx][:self.k]
            classes_counts = Y[ind_, :].sum(axis=0)
            majority = np.argsort(classes_counts)[::-1][0]
            result[idx] = {"knn_idx": ind_, "MJ": majority}
        return result

    def InitTypes(self, Y: np.ndarray):
        T = self.C.copy()
        T[T < 0.3] = self.minority_class["SF"]
        T[(T >= 0.3) & (T < 0.7)] = self.minority_class["BD"]
        T[(T >= 0.7) & (T < 1)] = self.minority_class["RR"]
        T[T == 1] = self.minority_class["OT"]

        for row, dic in self.knn_.items():
            maj_class = dic["MJ"]
            if Y[row, maj_class] == 1:
                T[row, maj_class] = 1

        while True:
            change_check = False
            for i in range(self.num_samples):
                for j in range(self.num_classes):
                    if T[i, j] == self.minority_class["RR"]:
                        for nearest in self.knn_[i]["knn_idx"]:
                            if T[nearest, j] in [self.minority_class["SF"], self.minority_class["BD"]]:
                                T[i, j] = self.minority_class["BD"]
                                change_check = True
                                break
            if not change_check:
                return T

    def seed_ref_instance(self):
        seed_int = np.random.choice(list(range(self.num_samples)), 1, p=self.w)[0]
        ref_int = self.knn_[seed_int]["knn_idx"][np.random.randint(self.k, size=1)[0]]
        return seed_int, ref_int

    def GenerateInstance(self, seed_int: int, ref_int: int, X: np.ndarray, Y: np.ndarray):
        X_s, Y_s = X[seed_int, :], Y[seed_int, :]
        X_r, Y_r = X[ref_int, :], Y[ref_int, :]
        X_c = np.random.uniform(low=0, high=1, size=self.num_features) * (X_r - X_s)
        Y_c = np.zeros(self.num_classes)

        d_s = np.linalg.norm(X_c - X_s, ord=2)
        d_r = np.linalg.norm(X_c - X_r, ord=2)
        cd = d_s / (d_s + d_r + 1e-6)

        for j in range(self.num_classes):
            if Y_s[j] == Y_r[j]:
                Y_c[j] = Y_s[j]
            else:
                if self.T[seed_int, j] == self.minority_class["MJ"]:
                    seed_int, ref_int = ref_int, seed_int
                    cd = 1 - cd

                theta = 0.5
                if self.T[seed_int, j] == self.minority_class["BD"]:
                    theta = 0.75
                elif self.T[seed_int, j] == self.minority_class["RR"]:
                    theta = 1.00001
                elif self.T[seed_int, j] == self.minority_class["OT"]:
                    theta = -0.00001

                Y_c[j] = Y_s[j] if cd <= theta else Y_r[j]

        return X_c, Y_c

mlsol = MLSol()

# Use dense arrays for the custom implementation.
X_train_dense = X_train.toarray()
Y_train_dense = y_train

X_train_mlsol, Y_train_mlsol = mlsol.oversample(X_train_dense, Y_train_dense, P=0.5, k=5)
print("MLSOL shapes:", X_train_mlsol.shape, Y_train_mlsol.shape)

clf_after_mlsol = OneVsRestClassifier(CalibratedClassifierCV(LinearSVC()))
clf_after_mlsol.fit(X_train_mlsol, Y_train_mlsol)
mlsol_pred = clf_after_mlsol.predict(X_test)

print_score(y_test, mlsol_pred, clf_name="After MLSOL")



## 13. Saving reusable artifacts

For the GitHub repository, save the model artifacts used by your website backend in a separate folder such as `backend/ml_artifacts/` or `models/`.

Typical files:
- TF-IDF vectorizer
- MultiLabelBinarizer
- trained classifier


In [ ]:

# Example only — uncomment and adapt when you are ready to save artifacts.
# import joblib
# MODELS_DIR.mkdir(parents=True, exist_ok=True)
# joblib.dump(tag_vectorizer, MODELS_DIR / "tfidf_vectorizer.joblib")
# joblib.dump(mlb, MODELS_DIR / "multilabel_binarizer.joblib")
# joblib.dump(calibrated_clf, MODELS_DIR / "baseline_classifier.joblib")



## 14. Repository note

For GitHub, this notebook can live alongside the website code as a separate research appendix notebook.

Recommended naming:
- `thesis_tag_prediction_research_clean.ipynb` — presentation-friendly version
- `thesis_tag_prediction_research_with_sampling.ipynb` — research appendix with MLkNN, MLSMOTE, and MLSOL
